# 02 — Image Processor

Generates only missing scene images from `scenes.json` using Stable Diffusion XL Base. Jobs resume from Google Drive.


In [ ]:
# ============================================================
# SETUP — REFRESH REPOSITORY
# ============================================================

import os
import sys
import subprocess

ROOT = "/content/black-history-factory"
REPO_URL = "https://github.com/jonbBla/black-history-factory.git"

if os.path.exists(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "origin/HEAD"], check=True)
    subprocess.run(["git", "-C", ROOT, "clean", "-fd"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

for name in list(sys.modules):
    if name.startswith("factory"):
        del sys.modules[name]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "diffusers",
        "transformers",
        "accelerate",
        "safetensors",
    ],
    check=True,
)

commit = subprocess.check_output(
    ["git", "-C", ROOT, "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print(f"[SETUP] Repository: {ROOT}")
print(f"[SETUP] Commit: {commit}")


In [ ]:
# ============================================================
# GOOGLE DRIVE + CONFIG
# ============================================================

from factory.drive import mount_drive, DrivePaths
from factory.config import Config

MYDRIVE = mount_drive()
paths = DrivePaths(
    os.path.join(MYDRIVE, "BLACK_HISTORY_FACTORY")
)

paths.ensure_tree()
config = Config.load(paths.root)

print(f"[SETUP] Drive: {paths.root}")
print(f"[IMAGE] Model: {config.image_model}")
print(f"[IMAGE] Size: {config.image_width}x{config.image_height}")
print(f"[IMAGE] Steps: {config.image_steps}")
print(f"[IMAGE] Guidance: {config.image_guidance_scale}")


In [ ]:
# ============================================================
# LOAD STABLE DIFFUSION XL BASE
# ============================================================

import torch
from diffusers import StableDiffusionXLPipeline

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
    add_watermarker=False,
)

pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print("[IMAGE] Stable Diffusion XL Base loaded.")


In [ ]:
# ============================================================
# INSPECT FIRST AVAILABLE JOB
# ============================================================

from factory.utils import read_json

jobs = []

if paths.jobs_dir.exists():
    for jid_path in sorted(paths.jobs_dir.iterdir()):
        if not jid_path.is_dir():
            continue

        jid = jid_path.name
        manifest = read_json(paths.manifest(jid), {}) or {}

        if manifest.get("status") in (
            "QWEN_READY",
            "IMAGES_PARTIAL",
        ):
            jobs.append(jid)

if not jobs:
    print("[IMAGE] No QWEN_READY or IMAGES_PARTIAL job found.")
else:
    jid = jobs[0]
    scenes = read_json(paths.scenes(jid), []) or []

    print(f"[IMAGE] Job: {jid}")
    print(f"[IMAGE] Scenes: {len(scenes)}")

    if scenes:
        print("\n[IMAGE] First image prompt:\n")
        print(scenes[0].get("image_prompt", ""))


In [ ]:
# ============================================================
# PROCESS ONE JOB
# ============================================================

from factory.image_engine import run
from factory.utils import read_json, write_json_atomic
from factory import status


def process_one():
    jobs = []

    # Find jobs waiting for image generation.
    if paths.jobs_dir.exists():
        for jid_path in sorted(paths.jobs_dir.iterdir()):
            if not jid_path.is_dir():
                continue

            jid = jid_path.name
            manifest = read_json(
                paths.manifest(jid),
                {}
            ) or {}

            if manifest.get("status") in (
                "QWEN_READY",
                "IMAGES_PARTIAL",
            ):
                jobs.append(jid)

    if not jobs:
        print("[IMAGE] No QWEN_READY job.")
        return False

    jid = jobs[0]

    scenes = read_json(
        paths.scenes(jid),
        []
    ) or []

    if not scenes:
        print(
            f"[IMAGE] ERROR {jid} | "
            "scenes.json is empty."
        )
        return False

    print(f"[IMAGE] START {jid}")
    print(f"[IMAGE] Scenes: {len(scenes)}")

    status.set_processor(
        paths,
        "image",
        "running",
        jid,
        "image_generation",
        f"0/{len(scenes)}",
        0,
        len(scenes),
    )

    def progress(n, total):
        print(f"[IMAGE] Scene {n}/{total}")

        status.set_processor(
            paths,
            "image",
            "running",
            jid,
            "image_generation",
            f"scene {n}/{total}",
            n,
            total,
        )

    try:
        run(
            paths,
            jid,
            scenes,
            pipe,
            config,
            progress,
        )

        manifest = read_json(
            paths.manifest(jid),
            {}
        ) or {}

        manifest["status"] = "IMAGES_READY"

        write_json_atomic(
            paths.manifest(jid),
            manifest
        )

        status.set_processor(
            paths,
            "image",
            "idle",
            jid,
            "ready",
            "images complete",
            len(scenes),
            len(scenes),
        )

        print(f"[IMAGE] COMPLETE {jid}")
        return True

    except Exception as e:
        manifest = read_json(
            paths.manifest(jid),
            {}
        ) or {}

        manifest.update(
            status="IMAGES_PARTIAL",
            image_error=str(e),
        )

        write_json_atomic(
            paths.manifest(jid),
            manifest
        )

        status.set_processor(
            paths,
            "image",
            "error",
            jid,
            "failed",
            str(e),
        )

        print(f"[IMAGE] ERROR {jid} | {e}")

        # Keep the full traceback visible in Colab.
        raise


# Run one job first.
process_one()


In [ ]:
# ============================================================
# OPTIONAL — PROCESS ALL AVAILABLE JOBS
# Run this only after the single-job test succeeds.
# ============================================================

while process_one():
    pass
